# Fetal brain US — stratified grouped 5-fold CV (full pipeline)
Runtime → **T4 GPU**, then *Run all*. Only cell 2 needs your input (the dataset zip in Drive).
Pipeline: unzip → design classes from real case counts → 5 stratified grouped folds → train (regularised) →
evaluate (CIs, real baseline) → frozen-feature probe → back up to Drive.

In [1]:
# 1. GPU + Drive
import tensorflow as tf, subprocess, sys, os, glob, zipfile, shutil, json
print('GPU:', tf.config.list_physical_devices('GPU'))
from google.colab import drive
drive.mount('/content/drive')

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mounted at /content/drive


In [2]:
# 2. Dataset zip in Drive  (set ZIP, or leave None to search)
ZIP = None   # e.g. '/content/drive/MyDrive/FetalBrain_XAI_Workspace/dataset.zip'
if ZIP is None:
    cands = glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True)
    print('zip files found:'); [print('  ', c) for c in cands]
    key = [c for c in cands if any(k in c.lower() for k in ('fetal', 'foetal', 'brain', 'ultrasound', 'dataset', 'roboflow'))]
    assert len(key) == 1, 'set ZIP above to one of the paths listed'
    ZIP = key[0]
print('using', ZIP)
shutil.rmtree('/content/ds_raw', ignore_errors=True)
zipfile.ZipFile(ZIP).extractall('/content/ds_raw')
# the export root is the folder that contains train/ (with _classes.csv or class folders)
RAW = next(os.path.dirname(p) for p in glob.glob('/content/ds_raw/**/train', recursive=True))
print('RAW =', RAW, {s: len(os.listdir(f'{RAW}/{s}')) for s in ('train', 'valid', 'test') if os.path.isdir(f'{RAW}/{s}')})

zip files found:
   /content/drive/MyDrive/FetalBrain_XAI_Workspace/05_Model_Backups/experiment2_backup.zip
   /content/drive/MyDrive/FetalBrain_XAI_Workspace/05_Model_Backups/models_backup (1).zip
   /content/drive/MyDrive/FetalBrain_XAI_Workspace/05_Model_Backups/experiment1_backup.zip
   /content/drive/MyDrive/FetalBrain_XAI_Workspace/02_Original_Dataset/Fetal Brain Abnormalities Ultrasound.v1i.multiclass.zip


AssertionError: set ZIP above to one of the paths listed

In [ ]:
# 3. Code from GitHub (always the latest)
if os.path.exists('/content/repo'):
    subprocess.run(['git', '-C', '/content/repo', 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/MUKUL-PRASAD-SIGH/XAI', '/content/repo'], check=True)
S = '/content/repo/scripts'
def run(*a):
    r = subprocess.run([sys.executable, *map(str, a)], capture_output=True, text=True)
    print(r.stdout[-7000:]); print(r.stderr[-1500:] if r.returncode else '')
    assert r.returncode == 0, 'step failed'

In [ ]:
# 4. Audit (leakage / duplicates / sources) — read it, it explains the dataset
run(f'{S}/01_dataset_audit.py', '--src', RAW, '--out', '/content/audit')

In [ ]:
# 5. Design the classes from the REAL number of independent cases per diagnosis
run(f'{S}/02c_design_classes.py', '--src', RAW, '--out', '/content/class_map_auto.json', '--min-groups', 5)

In [ ]:
# 6. Stratified grouped 5-fold dataset
CV = '/content/dataset_cv'
shutil.rmtree(CV, ignore_errors=True)
run(f'{S}/02b_make_cv_folds.py', '--src', RAW, '--out', CV, '--folds', 5,
    '--class-map', '/content/class_map_auto.json', '--min-groups', 5, '--link')

In [ ]:
# 7. Train all 5 folds (dropout .5, mixup, cutout, grayscale, AdamW, label smoothing, val_loss stopping)
r = subprocess.run([sys.executable, f'{S}/03_train.py', '--cv', CV, '--model', 'xception',
                    '--out', '/content/models_cv', '--mixed-precision'],
                   capture_output=True, text=True)
print('return code', r.returncode)
print(r.stdout[-3000:])
if r.returncode:
    print('--- STDERR ---'); print(r.stderr[-6000:])
assert r.returncode == 0, 'training failed (see STDERR above)'


In [ ]:
# 8. Evaluate pooled CV + frozen-feature probe on the same folds
run(f'{S}/04_evaluate.py', '--cv', CV, '--models', '/content/models_cv', '--out', '/content/results_cv', '--tta')
run(f'{S}/13_cv_probe.py', '--cv', CV, '--out', '/content/results_probe')
print(open('/content/results_cv/results_table.md').read())

In [ ]:
# 9. Optional second pass: also merge classes the model itself confuses, then redo 6-8
# run(f'{S}/02c_design_classes.py', '--src', RAW, '--out', '/content/class_map_auto2.json',
#     '--predictions', '/content/results_cv/predictions_xception.csv')

In [ ]:
# 10. Back up everything to Drive
OUT = '/content/drive/MyDrive/FetalBrain_XAI_Workspace/cv_run'
os.makedirs(OUT, exist_ok=True)
for d in ('models_cv', 'results_cv', 'results_probe', 'audit'):
    shutil.copytree(f'/content/{d}', f'{OUT}/{d}', dirs_exist_ok=True)
for f in ('class_map_auto.json', 'class_map_auto.report.txt'):
    shutil.copy(f'/content/{f}', OUT)
shutil.copy(f'{CV}/cv_report.txt', OUT); shutil.copy(f'{CV}/cv_manifest.csv', OUT)
print('saved to', OUT)